# 🏭 Project 3.4 — Predictive Maintenance Flag
**DSA for Mechatronics | Week 03 Project | ⭐ Recommended for Y3**

---

> **Submission:** Upload your completed `.ipynb` to BlackBoard before the deadline.  
> **Presentation:** You will run and explain your code live in class the following session.  
> **AI tools:** Allowed. You must understand and be able to explain every line you submit.

---

## 🎯 What you are building

A CNC milling machine has a vibration sensor on the spindle. It records RMS vibration (mm/s)
every second for 10 minutes — 600 readings total.

The signal contains:
- **Normal background vibration**: 0.5–2.5 mm/s  
- **Two fault windows** where vibration is sustained above 5 mm/s (real component wear)  
- **Periodic bearing spikes** appearing every N seconds — the bearing period is hidden in the data

A factory maintenance engineer needs three things from you:

1. An efficient way to answer "what was the average vibration between time A and B?" — **O(1) per query** using prefix sums  
2. Automatic detection of all sustained fault events — **sliding window**  
3. An estimate of the bearing fault period — **frequency counting on spike gaps**  

---

## 📐 Week 03 concepts — and why they matter here

| Concept | Where used | Why it matters |
|---|---|---|
| **Prefix sums** | Exercise 1 | O(1) range queries vs O(n) — critical at scale |
| **Sliding window** | Exercise 2 | Detect contiguous fault windows in one O(n) pass |
| **Frequency counting** | Exercise 4 | Find most common gap between periodic spikes |
| **String formatting** | Exercise 3 | Human-readable timestamped alarm report |

---

## 🔬 The prefix sum in depth

This is the most important algorithmic concept in the project.

```
signal  = [1.2, 1.1, 1.4, 8.5, 9.1, 8.8, 1.3, 1.2, ...]
                                ^fault^

prefix  = [0,   1.2, 2.3, 3.7, 12.2, 21.3, 30.1, 31.4, 32.6, ...]
```

**Without prefix sum** — to get average from t=100 to t=200:
```python
avg = sum(signal[100:201]) / 101    # scans 101 elements: O(n)
```

**With prefix sum** — same query:
```python
avg = (prefix[201] - prefix[100]) / 101    # one subtraction: O(1)
```

If you need to answer **10,000 such queries**, the difference is:
- Without: 10,000 × 101 = **1,010,000 operations**
- With: 10,000 × 1 = **10,000 operations** (after O(n) build cost)

This is why industrial monitoring systems always pre-compute prefix sums.

---

## ⚙️ Step 0 — Generate vibration data

Run this cell first. Do **not** modify it.  
The bearing period is hidden — you'll recover it in Exercise 4.


In [ ]:
import random
random.seed(2024)

N = 600                                     # 10 minutes at 1 Hz
BEARING_PERIOD = random.randint(15, 35)    # hidden — find it in Exercise 4

vibration = []
for t in range(N):
    # Normal background noise
    base = max(0.1, 1.0 + random.gauss(0, 0.35))
    
    # Periodic bearing spike (every BEARING_PERIOD seconds)
    bearing = random.uniform(4.0, 8.0) if t % BEARING_PERIOD == 0 else 0
    
    # Sustained fault windows (injected at known times — these are what Exercise 2 finds)
    fault = 0
    if 180 <= t <= 195:  fault = random.uniform(5.5, 9.0)
    if 380 <= t <= 392:  fault = random.uniform(6.0, 10.0)
    
    vibration.append(round(base + bearing + fault, 3))

print(f"Vibration signal: {N} samples ({N/60:.0f} min)")
print(f"Normal range: ~0.5 – 2.5 mm/s")
print(f"Samples above 5 mm/s (fault level): {sum(1 for v in vibration if v > 5)}")
print()
print("First 20 readings (mm/s):")
for t, v in enumerate(vibration[:20]):
    bar = '█' * int(v * 4)
    flag = ' ← SPIKE' if v > 4 else ''
    print(f"  t={t:03d}: {v:6.3f}  {bar}{flag}")
print("  ...")
print()
print("Hint: the bearing period is between 15 and 35 seconds — you'll estimate it in Exercise 4.")


---

## ✏️ Exercise 1 — Build the prefix sum + O(1) range queries

### Part A: `build_prefix_sum(signal)`
Returns a prefix array of length `len(signal) + 1`:
```
prefix[0]   = 0
prefix[i+1] = prefix[i] + signal[i]
```

### Part B: `range_average(prefix, a, b)`
Returns the mean of `signal[a..b]` (inclusive) in **O(1)**:
```
mean = (prefix[b+1] - prefix[a]) / (b - a + 1)
```

### Why `prefix[b+1] - prefix[a]`?
```
prefix[b+1] = signal[0] + signal[1] + ... + signal[b]
prefix[a]   = signal[0] + signal[1] + ... + signal[a-1]

difference  = signal[a] + signal[a+1] + ... + signal[b]   ← exactly what we want
```

> 💡 **Building prefix with a loop:**
> ```python
> prefix = [0]
> for v in signal:
>     prefix.append(prefix[-1] + v)
> return prefix
> ```
> `prefix[-1]` = last element. Each iteration adds the next signal value to the running total.

> ⚠️ **Edge case:** if `a > b`, raise a `ValueError` or return `0.0`.


In [ ]:
def build_prefix_sum(signal):
    """
    Build prefix sum array.
    
    prefix[0]   = 0
    prefix[i+1] = prefix[i] + signal[i]
    
    Length = len(signal) + 1
    """
    prefix = [0]
    for v in signal:
        # YOUR CODE: append prefix[-1] + v
        pass
    return prefix


def range_average(prefix, a, b):
    """
    Return mean of signal[a..b] (inclusive) in O(1).
    
    Uses: (prefix[b+1] - prefix[a]) / (b - a + 1)
    """
    if a > b:
        return 0.0
    # YOUR CODE: one line using the formula above
    pass


# ── Test ────────────────────────────────────────────────────────────────
prefix = build_prefix_sum(vibration)

print(f"Prefix array length : {len(prefix)}  (should be {len(vibration)+1})")
print()

# Test range queries
queries = [
    (  0,  59, "minute 1 (normal)"),
    ( 60, 119, "minute 2 (normal)"),
    (180, 195, "fault window 1  "),
    (380, 392, "fault window 2  "),
    (  0, 599, "full recording  "),
]

print(f"{'Range':<25} {'Avg vibration':>15}  {'Status'}")
print("─" * 60)
for a, b, label in queries:
    avg = range_average(prefix, a, b)
    status = "⚠️  FAULT" if avg > 3.5 else "✅ normal"
    print(f"  t={a:03d}–{b:03d}  {label}  {avg:>8.3f} mm/s   {status}")

# Verify against direct computation (should match)
direct = sum(vibration[0:60]) / 60
fast   = range_average(prefix, 0, 59)
print(f"\nVerification: direct={direct:.4f}  prefix={fast:.4f}  match={abs(direct-fast)<1e-9}")


---

## ✏️ Exercise 2 — Detect fault windows (Sliding Window)

Write `find_fault_windows(signal, threshold, min_duration)` that returns a list of fault events.

A **fault event** is a contiguous run of `min_duration` or more readings all **above** `threshold`.

### Algorithm:
```
i = 0
while i < len(signal):
    if signal[i] > threshold:          ← start of a potential fault
        j = i
        while j < len(signal) and signal[j] > threshold:
            j += 1                      ← extend to end of run
        duration = j - i
        if duration >= min_duration:
            record this event
        i = j                           ← jump past the run
    else:
        i += 1
```

### Each event is a dict:
```python
{
    'start':    180,           # first index above threshold
    'end':      195,           # last index above threshold
    'duration': 16,            # number of readings
    'peak':     9.84,          # max value in window
    'avg':      7.23           # mean value in window
}
```

### Expected result (with threshold=5.0, min_duration=5):
```
Found 2 fault event(s):
  Event 1: t=180–195  (16 s)  peak=9.84 mm/s  avg=7.23 mm/s
  Event 2: t=380–392  (13 s)  peak=9.95 mm/s  avg=7.81 mm/s
```

> 💡 **Computing peak and avg for the window:**
> ```python
> window = signal[i : j]                       # slice the fault window
> peak   = max(window)
> avg    = sum(window) / len(window)
> ```

> ⚠️ **Note:** use `threshold=5.0` and `min_duration=5` for the test.
> The bearing spikes (< 1 second each) should NOT appear as fault events since they are isolated.


In [ ]:
def find_fault_windows(signal, threshold=5.0, min_duration=5):
    """
    Detect sustained high-vibration fault events.
    
    Parameters
    ----------
    signal       : list of float
    threshold    : float — vibration level to trigger alarm (mm/s)
    min_duration : int   — minimum consecutive readings above threshold
    
    Returns
    -------
    list of dict, each with keys: start, end, duration, peak, avg
    """
    events = []
    i = 0
    
    while i < len(signal):
        if signal[i] > threshold:
            # Find end of this run
            j = i
            while j < len(signal) and signal[j] > threshold:
                j += 1
            
            duration = j - i
            
            if duration >= min_duration:
                window = signal[i : j]
                events.append({
                    'start':    i,
                    'end':      j - 1,
                    'duration': duration,
                    'peak':     round(max(window), 3),
                    'avg':      round(sum(window) / len(window), 3)
                })
            
            i = j  # jump past this run
        else:
            # YOUR CODE: advance i by 1 when no fault
            pass
    
    return events


# ── Test ────────────────────────────────────────────────────────────────
events = find_fault_windows(vibration, threshold=5.0, min_duration=5)

print(f"Found {len(events)} fault event(s):\n")
for k, e in enumerate(events, 1):
    print(f"  Event {k}: t={e['start']:03d}–{e['end']:03d}  "
          f"({e['duration']} s)  "
          f"peak={e['peak']:.2f} mm/s  "
          f"avg={e['avg']:.2f} mm/s")


---

## ✏️ Exercise 3 — Maintenance report (String formatting)

Write `maintenance_report(signal, events, prefix)` that prints a complete report.

### Target format:
```
╔══════════════════════════════════════════════════════╗
║          CNC SPINDLE — MAINTENANCE REPORT           ║
╚══════════════════════════════════════════════════════╝

  Recording   : 600 s  (10 min 00 s)
  Overall avg : 1.823 mm/s
  Normal range: 0.5 – 2.5 mm/s
  Threshold   : 5.0 mm/s  (sustained ≥ 5 s)

  ─── Fault Events ────────────────────────────────────
  Event 1  |  03:00 – 03:15  |  16 s  |  peak 9.84 mm/s
  Event 2  |  06:20 – 06:32  |  13 s  |  peak 9.95 mm/s

  Total time in fault: 29 s (4.8% of recording)

  ─── Machine Status ──────────────────────────────────
  🔴 FAULT  — 2 sustained fault event(s) detected
```

**Status rules:**
- 🟢 OK      — 0 events
- 🟡 CAUTION — 1 event
- 🔴 FAULT   — 2 or more events

> 💡 **Timestamp formatting (seconds → MM:SS):**
> ```python
> def fmt_time(t_seconds):
>     m = t_seconds // 60
>     s = t_seconds % 60
>     return f"{m:02d}:{s:02d}"
> ```
> So `t=195` → `fmt_time(195)` = `"03:15"`.

> 💡 **Total fault time:**
> ```python
> total_fault = sum(e['duration'] for e in events)
> fault_pct   = total_fault / len(signal) * 100
> ```


In [ ]:
def fmt_time(t_seconds):
    """Convert seconds to MM:SS string.  e.g. 195 → '03:15' """
    m = t_seconds // 60
    s = t_seconds % 60
    return f"{m:02d}:{s:02d}"


def maintenance_report(signal, events, prefix, threshold=5.0):
    """Print formatted CNC maintenance report."""
    
    n           = len(signal)
    overall_avg = range_average(prefix, 0, n - 1)
    total_fault = sum(e['duration'] for e in events)
    fault_pct   = total_fault / n * 100
    
    if len(events) == 0:   status = "🟢 OK"
    elif len(events) == 1: status = "🟡 CAUTION"
    else:                  status = "🔴 FAULT"
    
    # YOUR CODE: print the report in the format shown above
    # Key pieces:
    #   fmt_time(t)                 → "MM:SS" string
    #   range_average(prefix, a, b) → average in a window (O(1))
    #   "─" * n                     → horizontal divider
    #   f"{value:<20}"              → left-align in 20 chars
    
    width = 54
    print("╔" + "═"*width + "╗")
    print("║" + "CNC SPINDLE — MAINTENANCE REPORT".center(width) + "║")
    print("╚" + "═"*width + "╝")
    print()
    print(f"  Recording   : {n} s  ({fmt_time(n)})")
    print(f"  Overall avg : {overall_avg:.3f} mm/s")
    # ... continue


# ── Run ─────────────────────────────────────────────────────────────────
prefix = build_prefix_sum(vibration)
events = find_fault_windows(vibration, threshold=5.0, min_duration=5)
maintenance_report(vibration, events, prefix)


---

## ✏️ Exercise 4 — Bearing period detection (Frequency Counting)

A loose bearing creates a vibration spike every `BEARING_PERIOD` seconds.
You don't know what that period is — you need to estimate it from the data.

### Algorithm:
1. Find all **spike indices**: positions where `vibration[t] > spike_threshold` (use 3.5)
2. Compute all **gaps** between consecutive spikes: `gap[i] = spikes[i+1] - spikes[i]`
3. **Count** how often each gap size appears (frequency dict)
4. The **most common gap** is the estimated bearing period

### Why this works:
```
Real period = 20 s
Spike indices might be: [20, 40, 60, 80, 100, ...]
Gaps: [20, 20, 20, 20, ...]  → most common gap = 20 ✅

With noise (some spikes missed, fault windows add extra spikes):
Gaps: [20, 20, 40, 20, 20, 20, 16, 20, ...]
Most common is still 20 ✅ (noise-robust)
```

> 💡 **Finding spike indices:**
> ```python
> spikes = [t for t, v in enumerate(signal) if v > spike_threshold]
> ```

> 💡 **Computing gaps:**
> ```python
> gaps = [spikes[i+1] - spikes[i] for i in range(len(spikes) - 1)]
> ```

> 💡 **Frequency count (same pattern as Exercise 2 in P3.2):**
> ```python
> freq = {}
> for g in gaps:
>     freq[g] = freq.get(g, 0) + 1
> most_common_gap = max(freq, key=freq.get)
> ```


In [ ]:
def detect_bearing_period(signal, spike_threshold=3.5):
    """
    Estimate the period of a periodic bearing fault spike.
    
    Steps:
    1. Find all t where signal[t] > spike_threshold
    2. Compute gaps between consecutive spike indices
    3. Frequency-count the gaps
    4. Return the most common gap (= estimated period)
    
    Returns
    -------
    int — estimated period in seconds
    """
    # Step 1: find spike indices
    spikes = [t for t, v in enumerate(signal) if v > spike_threshold]
    
    if len(spikes) < 2:
        return None   # not enough spikes to estimate
    
    # Step 2: compute gaps
    gaps = [spikes[i+1] - spikes[i] for i in range(len(spikes) - 1)]
    
    # Step 3: frequency count
    freq = {}
    # YOUR CODE: count each gap
    
    # Step 4: return most common gap
    # YOUR CODE: return max(freq, key=freq.get)
    pass


# ── Test ────────────────────────────────────────────────────────────────
estimated = detect_bearing_period(vibration, spike_threshold=3.5)

print(f"Estimated bearing period : {estimated} s")
print(f"Actual bearing period    : {BEARING_PERIOD} s   (hidden variable revealed!)")
print(f"Error                    : {abs(estimated - BEARING_PERIOD)} s")
print()

# Show gap frequency distribution
spikes = [t for t, v in enumerate(vibration) if v > 3.5]
gaps   = [spikes[i+1] - spikes[i] for i in range(len(spikes)-1)]
freq   = {}
for g in gaps:
    freq[g] = freq.get(g, 0) + 1

print("Gap frequency distribution:")
for gap in sorted(freq.keys()):
    bar = '█' * freq[gap]
    print(f"  gap={gap:>3} s : {freq[gap]:>3}×  {bar}")


---

## 🚀 Stretch Goal — ASCII Vibration Chart

Print a time-series bar chart of the vibration signal sampled every 5 seconds,
with fault windows highlighted.

```python
fault_set = set()
for e in events:
    for t in range(e['start'], e['end'] + 1):
        fault_set.add(t)

print("Vibration chart (sampled every 5 s)")
print("Scale: each █ = 0.5 mm/s")
print()
for t in range(0, len(vibration), 5):
    v = vibration[t]
    bar_len = int(v * 2)
    bar = '█' * bar_len
    fault_marker = ' ⚠️' if t in fault_set else ''
    print(f"  {fmt_time(t)} | {bar:<20} {v:.2f}{fault_marker}")
```
